In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from src.preprocess import build_preprocessor
from src.metrics import evaluate

cols = model_features(df)                 # SEX, AGE, MARRIAGE nikal kar
Xtr = X_train[cols]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Majority baseline": DummyClassifier(strategy="prior"),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "Logistic (class_weight=balanced)": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

rows = []
for name, clf in models.items():
    pipe = make_pipeline(build_preprocessor(cols), clf)
    proba = cross_val_predict(pipe, Xtr, y_train, cv=cv, method="predict_proba")[:, 1]
    rows.append({"model": name, **evaluate(y_train, proba, threshold=0.5)})

pd.DataFrame(rows)[["model", "roc_auc", "pr_auc", "precision", "recall", "f1", "tn", "fp", "fn", "tp"]]

,model,roc_auc,pr_auc,precision,recall,f1,tn,fp,fn,tp
0,Majority baseline,0.4999,0.2212,0.0000,0.0000,0.0000,18691,0,5309,0
1,Logistic regression,0.7264,0.5060,0.7219,0.2445,0.3653,18191,500,4011,1298
2,Logistic (class_weight=balanced),0.7265,0.5045,0.3732,0.6642,0.4778,12768,5923,1783,3526


In [10]:
pipe = make_pipeline(build_preprocessor(cols), LogisticRegression(max_iter=1000))
proba_lr = cross_val_predict(pipe, Xtr, y_train, cv=cv, method="predict_proba")[:, 1]

rows = []
for t in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15]:
    m = evaluate(y_train, proba_lr, threshold=t)
    m["cost_r5"] = 5 * m["fn"] + m["fp"]
    m["cost_r1"] = m["fn"] + m["fp"]
    rows.append(m)

pd.DataFrame(rows)[["threshold", "precision", "recall", "f1", "fp", "fn", "cost_r1", "cost_r5"]]

,threshold,precision,recall,f1,fp,fn,cost_r1,cost_r5
0,0.50,0.7219,0.2445,0.3653,500,4011,4511,20555
1,0.40,0.6405,0.3899,0.4847,1162,3239,4401,17357
2,0.30,0.5775,0.4555,0.5093,1769,2891,4660,16224
3,0.25,0.4866,0.5511,0.5169,3087,2383,5470,15002
4,0.20,0.3201,0.7156,0.4423,8071,1510,9581,15621
5,0.15,0.2712,0.7937,0.4043,11325,1095,12420,16800
